# MAPL+ Pipeline — Step 1 to Step 3

| Step | Deskripsi | Referensi |
|------|-----------|-----------|
| 1 | Aggregate raw transactions → daily panel | — |
| 2 | Own-price elasticity per SKU (log-log OLS) | OpenStax Introductory Business Statistics, Ch. 13.5 |
| 3 | Cannibalization detection via Difference-in-Differences | Van Heerde et al. (2004), McColl et al. (2020), Varian (2016), Herrala (2018) |

## Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


---
## Step 1 — Aggregate ke Daily Panel

Aggregate raw transactions ke daily panel.  
Satu baris = **1 SKU × 1 hari × 1 branch**.

> Oktober (bulan 10) di-exclude sesuai preprocessing awal.

In [26]:
def step1_daily_panel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate raw transactions ke daily panel.
    Satu baris = 1 SKU × 1 hari × 1 branch.
    """
    # ← Hapus pd.read_csv dan pd.concat di sini, gunakan df parameter langsung

    df = df.copy()  # hindari modifikasi df asli
    df['Date'] = pd.to_datetime(df['Date'])

    # Exclude Oktober (bulan 10) sesuai preprocessing awal
    df = df[df['Date'].dt.month != 10]

    # Tambah WeekNum (0-indexed dari hari pertama)
    min_date = df['Date'].min()
    df['WeekNum'] = ((df['Date'] - min_date).dt.days // 7).astype(int)

    daily = (
        df.groupby(['Date', 'WeekNum', 'Branch', 'SKU_ID', 'SKU', 'Brand', 'SKU_Category'])
        .agg(
            DailyQty    = ('Qty',               'sum'),
            Price       = ('DiscountedPrice',   'first'),
            Discount    = ('DiscountPercentage','first'),
            IsPromo     = ('IsPromo',           'max'),
            NormalPrice = ('NormalPrice',       'first'),
        )
        .reset_index()
    )

    print(f'[Step 1] Daily panel: {len(daily):,} baris '
          f'({daily["Date"].nunique()} hari × '
          f'{daily["Branch"].nunique()} branch × '
          f'{daily["SKU_ID"].nunique()} SKU)')
    return daily

In [27]:
# ── Opsi A: mulai dari raw transactions ──
df1 = pd.read_csv('transaction_1_v5.csv')
df2 = pd.read_csv('transaction_2_v5.csv')
df  = pd.concat([df1, df2], ignore_index=True)
if 'Date' not in df.columns:
    df['Date'] = pd.to_datetime(df['DateTime']).dt.date
    df['Date'] = pd.to_datetime(df['Date'])
print(f'Raw data loaded: {len(df):,} rows\n')
daily = step1_daily_panel(df)
daily.to_csv('daily_panel.csv', index=False)
print('→ Saved: daily_panel.csv')

# ── Opsi B: daily panel sudah ada (skip step 1) ──
# DAILY_PATH = 'daily_panel.csv'  # ganti sesuai path
# daily = pd.read_csv(DAILY_PATH, parse_dates=['Date'])
# print(f'Daily panel loaded: {len(daily):,} rows')
# daily

Raw data loaded: 205,946 rows

[Step 1] Daily panel: 12,240 baris (153 hari × 4 branch × 20 SKU)
→ Saved: daily_panel.csv


---
## Step 2 — Own-Price Elasticity per SKU

Estimasi own-price elasticity per SKU menggunakan **log-log OLS**.

$$\ln(\text{DailyQty}) = \alpha + \beta \cdot \ln(\text{Price}) + \varepsilon$$

$\beta$ = own-price elasticity *(OpenStax Ch. 13.5, Case 4: log-log)*

Pooled across all 4 branches per SKU.

In [ ]:
def step2_own_elasticity(daily: pd.DataFrame) -> pd.DataFrame:
    """
    Estimasi own-price elasticity per SKU menggunakan log-log OLS.

    Model: ln(DailyQty) = α + β × ln(Price) + ε
    β = own-price elasticity (OpenStax Ch. 13.5, Case 4: log-log)

    Pooled across all 4 branches per SKU.
    """
    results = []

    for sku_id, grp in daily.groupby('SKU_ID'):
        grp = grp.copy()

        # Filter: harga dan qty harus > 0
        grp = grp[(grp['Price'] > 0) & (grp['DailyQty'] > 0)]
        if len(grp) < 30:
            continue

        grp['ln_qty']   = np.log(grp['DailyQty'])
        grp['ln_price'] = np.log(grp['Price'])

        try:
            model    = ols('ln_qty ~ ln_price', data=grp).fit()
            beta     = model.params['ln_price']
            pval     = model.pvalues['ln_price']
            r2       = model.rsquared
            sku_name = grp['SKU'].iloc[0]

            results.append({
                'SKU_ID':        sku_id,
                'SKU':           sku_name,
                'OwnElasticity': round(beta, 4),
                'p_value':       round(pval, 4),
                'R2':            round(r2, 4),
                'N_obs':         len(grp),
                'Source':        'ols_loglog',
            })
        except Exception as e:
            print(f'  [Step 2] SKU {sku_id} error: {e}')

    elasticity_df = pd.DataFrame(results)
    print(f'\n[Step 2] Elasticity estimated for {len(elasticity_df)} SKUs')
    return elasticity_df

In [ ]:
elasticity_df = step2_own_elasticity(daily)

elasticity_df.to_csv('elasticity_per_sku.csv', index=False)
print('→ Saved: elasticity_per_sku.csv')

elasticity_df[['SKU_ID', 'SKU', 'OwnElasticity', 'p_value', 'R2']]


[Step 2] Elasticity estimated for 20 SKUs
→ Saved: elasticity_per_sku.csv


,SKU_ID,SKU,OwnElasticity,p_value,R2
0,S001,Richeese Wafer Keju 50g,-1.4360,0.0000,0.0338
1,S002,Richoco Wafer Cokelat 50g,-0.6261,0.0022,0.0153
2,S003,Richeese Wafer Keju 10g Renceng,-1.0113,0.0002,0.0230
3,S004,Richoco Wafer Cokelat 10g Renceng,-1.1112,0.0000,0.0406
4,S005,Nextar Brownies Pie 40g,-1.1361,0.0000,0.0460
5,S006,Nextar Nastar Pie 30g,-0.9365,0.0000,0.0320
6,S007,Richeese Siip Keju 20g,-0.8255,0.0000,0.0273
7,S008,Richoco Ahh! Extruded 15g,-0.5087,0.0781,0.0051
8,S009,Richeese Mi Instan Keju Pedas,-0.8243,0.0005,0.0197
9,S010,Richeese Mi Instan Ramen Keju,-0.5369,0.0076,0.0116


---
## Step 3 — Cannibalization Detection (DiD)

Deteksi cannibalization antar SKU menggunakan **DiD estimator** (Varian, 2016).

$$\text{DiD}_{AB} = \overline{\text{Residual}_B \mid \text{IsPromo}_A=1} - \overline{\text{Residual}_B \mid \text{IsPromo}_A=0}$$

**Cannibalization coefficient** (Herrala, 2018):

$$C(A \to B) = \frac{|\text{DiD}_{AB}|}{\overline{\text{Uplift}_A \mid \text{IsPromo}_A=1}}$$

Cannibalization **confirmed** jika: $p < p_{\text{threshold}}$ **AND** $\text{DiD}_{AB} < 0$

In [ ]:
def step3_cannibalization(daily: pd.DataFrame,
                           p_threshold: float = 0.05,
                           min_promo_days: int = 10,
                           max_coef: float = 1.0) -> pd.DataFrame:
    """
    Deteksi cannibalization antar SKU menggunakan DiD estimator.

    Framework (Varian, 2016):
      DiD_AB = mean(Residual_B | IsPromo_A=1)
             - mean(Residual_B | IsPromo_A=0)

    Cannibalization coefficient (Herrala, 2018):
      C(A→B) = |DiD_AB| / mean(Uplift_A | IsPromo_A=1)
        → di-cap di max_coef (default 1.0) untuk menghindari
        rasio yang tidak masuk akal secara bisnis

    Cannibalization confirmed jika: p < p_threshold AND DiD_AB < 0
    """
    skus     = daily['SKU_ID'].unique()
    branches = daily['Branch'].unique()
    records  = []

    for branch in branches:
        branch_data = daily[daily['Branch'] == branch].copy()

        for sku_a in skus:
            # Ambil hari-hari promo SKU A di branch ini
            a_data = branch_data[branch_data['SKU_ID'] == sku_a][
                ['Date', 'WeekNum', 'IsPromo', 'DailyQty']
            ].copy()

            promo_days    = set(a_data[a_data['IsPromo'] == 1]['Date'])
            no_promo_days = set(a_data[a_data['IsPromo'] == 0]['Date'])

            if len(promo_days) < min_promo_days:
                continue  # Tidak cukup promo days untuk SKU A ini

            for sku_b in skus:
                if sku_a == sku_b:
                    continue

                b_data = branch_data[branch_data['SKU_ID'] == sku_b][
                    ['Date', 'WeekNum', 'DailyQty']
                ].copy()

                if len(b_data) < 30:
                    continue

                # ── Sub-step 3A: Baseline SKU B dari no-promo days ──
                # Counterfactual: demand B "seandainya tidak ada promosi A"
                # (Varian 2016: "counterfactual constructed using data
                #  from before/outside the treatment period")
                b_no_promo = b_data[b_data['Date'].isin(no_promo_days)].copy()

                if len(b_no_promo) < 10:
                    continue

                try:
                    baseline_model = ols(
                        'DailyQty ~ WeekNum', data=b_no_promo
                    ).fit()
                except Exception:
                    continue

                # Predict baseline untuk semua hari
                b_data = b_data.copy()
                b_data['BaselineDemand'] = baseline_model.predict(b_data)
                b_data['Residual_B']     = b_data['DailyQty'] - b_data['BaselineDemand']

                # ── Sub-step 3B: DiD Estimator ──
                # Treatment: hari SKU A promo → lihat Residual B
                # Control  : hari SKU A tidak promo → lihat Residual B
                residual_treatment = b_data[
                    b_data['Date'].isin(promo_days)
                ]['Residual_B'].dropna()

                residual_control = b_data[
                    b_data['Date'].isin(no_promo_days)
                ]['Residual_B'].dropna()

                if len(residual_treatment) < 5 or len(residual_control) < 5:
                    continue

                did_ab = residual_treatment.mean() - residual_control.mean()

                # ── Sub-step 3C: Significance test ──
                # H0: DiD_AB = 0, H1: DiD_AB < 0 (one-sided)
                _, p_one = stats.ttest_ind(
                    residual_treatment, residual_control,
                    equal_var=False, alternative='less'
                )
                # scipy "less" = H1: mean(treatment) < mean(control) — already one-sided

                # Cannibalization coefficient (Herrala, 2018)
                # C(A→B) = |DiD_AB| / mean(Uplift_A saat promo)
                a_promo_data = a_data[a_data['Date'].isin(promo_days)].copy()
                a_no_promo   = a_data[a_data['Date'].isin(no_promo_days)].copy()

                cannib_coef = np.nan
                
                if len(a_promo_data) > 0 and len(a_no_promo) > 0:
                    mean_uplift_a = (
                        a_promo_data['DailyQty'].mean() -
                        a_no_promo['DailyQty'].mean()
                    )
                    if mean_uplift_a > 0:
                        raw_coef    = abs(did_ab) / mean_uplift_a
                        cannib_coef = min(raw_coef, max_coef)  # cap di 1.0

                records.append({
                    "Branch":       branch,
                    'SKU_A':        sku_a,   # promotor
                    'SKU_B':        sku_b,   # potential victim
                    "DiD_AB":       round(did_ab, 4),
                    "p_value":      round(p_one, 4),
                    "Cannib_Coef":  round(cannib_coef, 4) if not np.isnan(cannib_coef) else np.nan,
                    "Confirmed":    (p_one < p_threshold) and (did_ab < 0),
                    "IsCapped":     (not np.isnan(cannib_coef)) and (raw_coef > max_coef),
                    "N_treatment":  len(residual_treatment),
                    "N_control":    len(residual_control),
                })

    cannib_df = pd.DataFrame(records)

    confirmed = cannib_df[cannib_df['Confirmed'] == True]
    
    print(f'\n[Step 3] Total pairs tested       : {len(cannib_df):,}')
    print(f'[Step 3] Cannibalization confirmed: {len(confirmed)} pairs '
          f'(p < {p_threshold}, DiD < 0)')
    print(f"[Step 3] Cannib_Coef capped at    : {max_coef}")

    return cannib_df

In [ ]:
cannib_df = step3_cannibalization(daily, p_threshold=0.05, min_promo_days=10)

cannib_df.to_csv('cannibalization_detail.csv', index=False)
print('→ Saved: cannibalization_detail.csv')

cannib_df[cannib_df['Confirmed'] == True].head(20)


[Step 3] Total pairs tested       : 1,520
[Step 3] Cannibalization confirmed: 293 pairs (p < 0.05, DiD < 0)
[Step 3] Cannib_Coef capped at    : 1.0
→ Saved: cannibalization_detail.csv


,Branch,SKU_A,SKU_B,DiD_AB,p_value,Cannib_Coef,Confirmed,IsCapped,N_treatment,N_control
2,Bandung,S001,S004,-2.9494,0.0069,0.3161,True,False,44,109
20,Bandung,S002,S003,-2.6971,0.0123,1.0000,True,True,51,102
21,Bandung,S002,S004,-2.0647,0.0409,1.0000,True,True,51,102
25,Bandung,S002,S008,-1.6663,0.0393,0.8456,True,False,51,102
28,Bandung,S002,S011,-1.7937,0.0277,0.9102,True,False,51,102
33,Bandung,S002,S016,-3.3472,0.0007,1.0000,True,True,51,102
34,Bandung,S002,S017,-1.5956,0.0218,0.8097,True,False,51,102
37,Bandung,S002,S020,-2.2624,0.0095,1.0000,True,True,51,102
39,Bandung,S003,S002,-2.2325,0.0066,0.2832,True,False,39,114
40,Bandung,S003,S004,-4.9218,0.0000,0.6244,True,False,39,114


---
## Cannibalization Matrix

Build **N×N matrix** dari hasil Step 3.  
`C[A][B]` = koefisien cannibalization SKU A terhadap SKU B, dirata-rata across branches.  
Hanya pair yang **confirmed** yang non-zero.

In [ ]:
def build_cannib_matrix(cannib_df: pd.DataFrame,
                         skus: list,
                         agg: str = 'mean') -> pd.DataFrame:
    """
    Build NxN cannibalization matrix dari hasil step3.
    C[A][B] = koefisien cannibalization SKU A terhadap SKU B,
              averaged across branches.
    Hanya pair yang confirmed yang non-zero.
    """
    confirmed = cannib_df[cannib_df['Confirmed'] == True].copy()

    if agg == 'mean':
        agg_df = (
            confirmed.groupby(['SKU_A', 'SKU_B'])['Cannib_Coef']
            .mean()
            .reset_index()
        )
    else:
        agg_df = (
            confirmed.groupby(['SKU_A', 'SKU_B'])['Cannib_Coef']
            .median()
            .reset_index()
        )

    matrix = pd.DataFrame(0.0, index=skus, columns=skus)
    for _, row in agg_df.iterrows():
        if row['SKU_A'] in skus and row['SKU_B'] in skus:
            matrix.loc[row['SKU_A'], row['SKU_B']] = round(row['Cannib_Coef'], 4)

    return matrix

In [ ]:
skus   = sorted(daily['SKU_ID'].unique())
matrix = build_cannib_matrix(cannib_df, skus, agg='mean')

matrix.to_csv('cannibalization_matrix.csv')
print('→ Saved: cannibalization_matrix.csv')
print(f'\nCannibalization matrix ({len(skus)}×{len(skus)}):')
matrix

→ Saved: cannibalization_matrix.csv

Cannibalization matrix (20×20):


,S001,S002,S003,S004,S005,S006,S007,S008,S009,S010,S011,S012,S013,S014,S015,S016,S017,S018,S019,S020
S001,0.0000,0.0000,0.0000,0.2934,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3347,0.0000
S002,0.6244,0.0000,0.8064,0.9049,0.0000,0.0000,0.6681,0.8456,0.0000,0.0000,0.7407,0.0000,0.0000,0.0000,0.7133,0.9931,0.6318,0.7584,0.0000,0.7460
S003,0.0000,0.2832,0.0000,0.4720,0.0000,0.3525,0.0000,0.4438,0.0000,0.2990,0.0000,0.2844,0.0000,0.0000,0.2942,0.0000,0.1837,0.0000,0.3603,0.2857
S004,0.0000,0.0000,0.0000,0.0000,0.0000,0.4823,0.0000,0.2514,0.3916,0.0000,0.2375,0.0000,0.5845,0.3400,0.0000,0.3460,0.0000,0.0000,0.0000,0.3419
S005,0.0000,0.0000,0.0000,0.0000,0.0000,0.3547,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1940
S006,0.0000,0.3299,0.3580,0.0000,0.9481,0.0000,0.0000,0.0000,0.0000,0.3267,0.3954,0.0000,0.0000,0.3803,0.5733,0.0000,0.0000,0.3838,0.5338,0.0000
S007,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2133,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.6375,0.0000
S008,0.0000,0.3114,0.0000,0.0000,0.8942,0.0000,0.0000,0.0000,0.0000,0.4801,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8766,0.0000
S009,0.0000,0.0000,0.0000,0.4064,0.0000,0.0000,0.3800,0.0000,0.0000,0.6964,0.0000,0.0000,0.0000,0.4536,0.0000,0.6042,0.4130,0.5233,0.0000,0.8465
S010,0.0000,0.0000,0.0000,0.0000,0.0000,0.4963,0.2937,0.0000,0.5925,0.0000,0.0000,0.4593,0.0000,0.3559,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


---
✅ **Step 1–3 selesai.**

Output files:
- `daily_panel.csv` — daily panel (Step 1)
- `elasticity_per_sku.csv` — own-price elasticity per SKU (Step 2)
- `cannibalization_detail.csv` — detail DiD per pair per branch (Step 3)
- `cannibalization_matrix.csv` — N×N cannibalization matrix (Step 3)